%md
#### Goal
- To create a dataobject to have CDD Case Question and Answer for all W&R Sourcesystems

#### Authors
- Abhishek.R.Jaiswal@rabobank.com
- Mahalakshmi.Vengateswaran@rabobank.com
- Prasad.gadidala@rabobank.com

##### Flow of Logic
- Reading GCOB,Legacy2,GCDS,GIC,KN1 Caseservice,Party_RiskModelInstanceQuestionAnswers dataobjects from GDP
- Fetching the Question and Answer details of GCOB Clients from GRAM
- Fetching the Question and Answer details of Legacy2 Clients from Legacy2_risk
- Fetching the Question and Answer details of GIC Clients from GRAM
- Combining all the Question and Answer related information from different sourcesystems
- Loading the final dataobject to SA RADAR Storage account

#### Note: 
  - Currently this notebook has scope of GCOB,Legacy2,GCDS,KN1 SourceSystems

##### Version & Changes

|     Developer |Date	   | PBI/Bug No |	Changes done |
|----------|----------|----------|----------|
| Prasad  Gadidala	 |11-Aug-2025 |13357503 |KN1 Cases Instance question and answers those are not aviabale in GRAM and        brefore feb 2024 data issue fix. 
| Mahalakshmi V | 28-Oct-2025 | 14044973 | Added QuestionIsMaterialFlag column 
| Mahalakshmi V | 20-Nov-2025 | 14165735 | Made notebook compatible for Historical Load
| Abhishek	 |10-Dec-2025  |14372938    | Extract RANZ data from GRAM 
| Prajit Tatari	 |11-Feb-2026 |15058674 |Adding MDM RANZ data from GDP  
|Rhea Gupta | 30-Mar-2026 |15580816 | Included RANZ in version 3
|Abhishek Jaiswal | 14-Apr-2026 |15734307 | Included RiskModelSourceSystem column
| Mahalakshmi V | 20-May-2026 | 16071492 | Included RANZ V4

#####READING FILES FROM GDP

In [0]:
import os
import pandas as pd
from datetime import datetime, timedelta
from pyspark.sql import functions as F

In [0]:
app_reg_app_id = os.environ['APP_REG_APP_ID']
ReadStorage = os.environ['GDP_STORAGE_NAME']
TenantId = os.environ['TENANT_ID']
GIC_ReadStorage = os.environ['GDP_SA_STORAGE_NAME']
environment=os.environ['ENV']
radar_datamodel_version_number=3
SARADAR='saradar'+environment

In [0]:
from RadarUtils import *

In [0]:
dbutils.widgets.text("Load_Date", "")
dbutils.widgets.text("RunType", "daily")  # default is daily

Load_Date = dbutils.widgets.get("Load_Date")
RunType = dbutils.widgets.get("RunType").lower()
if Load_Date:
    RunType="historical"
    
if not Load_Date:
    Load_Date = datetime.today().strftime('%Y%m%d')

print(f"Running for Load Date: {Load_Date}")

In [0]:
party_CDDcasequestionanswers_dataobject='Party_CDDCase_QuestionAnswer'

In [0]:
authenticate_storage_account(ReadStorage)
authenticate_storage_account(GIC_ReadStorage)
authenticate_storage_account('edlcorestdauprod0001') #RANZ_ReadStorage
authenticate_storage_account(SARADAR)

In [0]:
load_dts = f"EDL_LOAD_DTS={Load_Date}*"
print (load_dts)

In [0]:
# List of datasets from GDP
load_df = [
'party_case_client_details'
]

# Create TempView for each loading table
for Object in load_df:
    Read_GDP_Defined_DataObjects(Source='GCOB', Dataobject=Object, path_prefix='CaseService',Load_Date=Load_Date)

In [0]:
# List of dataobjects from GDP
load_df = [
'Party_RiskModelInstanceQuestionAnswers'
]

# Create TempView for each loading table
for Object in load_df:
    Read_GDP_Defined_DataObjects(Source='GCOB', Dataobject=Object, path_prefix='RISKMODEL',Load_Date=Load_Date)

In [0]:
# List of datasets from GDP
load_df =[
'Legacy2_case_client_details',
'Legacy2_risk']

# Create TempView for each loading table
for Object in load_df:
    Read_GDP_Defined_DataObjects(Source='Legacy2', Dataobject=Object,path_prefix='Legacy2',Load_Date=Load_Date)

df_Party_SystemIdentifier=spark.read.parquet(f'abfss://radardatamodel@{SARADAR}.dfs.core.windows.net/Party_SystemIdentifier/3/data/{load_dts}/*.parquet')

In [0]:
# print list of strings for loading spark dfs from GDP
load_df = pd.DataFrame({'GDPname':[
'pessoa'
]})
#gic_load_dts
# Create TempView for each loading table
for index, row in load_df.iterrows():
    Read_GDP_Defined_DataObjects(Source='GIC', Dataobject=row.GDPname,Load_Date=Load_Date)

In [0]:
# print list of strings for loading spark dfs from GDP
load_df = pd.DataFrame({'GDPname':[
'ADKYC_CONTRAPARTES'
,'ADKYC_CONTRAPARTES_COMPL'
,'ADRBB_CONTRAPARTES_GRAM'
,'ADRBB_CONTRAPARTES_PONTUACOES_ABAS'
,'ADRBB_CONTRAPARTES_QUESTOES_RESPOSTAS'
,'ADRBB_QUESTOES'
]})

# Create TempView for each loading table
for index, row in load_df.iterrows():
    Read_GDP_Defined_DataObjects(Source='KN1', Dataobject=row.GDPname,Load_Date=Load_Date)

In [0]:
df_Party_KN1Cases = spark.read.parquet(f"abfss://radardatamodel@{SARADAR}.dfs.core.windows.net/Party_KN1Cases/3/data/{load_dts}/*.parquet")
df_Party_KN1Cases.createOrReplaceTempView("KN1Cases")

In [0]:
load_df = [
    'c_b_party_xref'
,'c_lkp_cdd_risk_rating_xref'
,'c_b_due_diligence_xref'
,'c_b_contract_xref'
,'c_b_contr_rol_party_xref'
]
load_ranz_v4_rdm_tables(load_df, Load_Date)

#####TRANSFORMATION TO GET Party_CDDCase_QuestionAnswer OBJECT 

In [0]:
%sql
Create or replace temporary view Gcob_CaseClientDetails As
select * 
,case when ClientType = 'Legal Entity' then 'Legal Entity' 
      when ClientType in ('Natural Person','Natural Person acting in a Professional Capacity (NPPC)') then 'Natural Person' 
      else null end as Party_type,Case
      when ClientType = 'Legal Entity' then concat('LEC_', GcobId)
      when ClientType in (
        'Natural Person',
        'Natural Person acting in a Professional Capacity (NPPC)'
      ) then concat('NP_NPPC_', GcobId)
    End as UniquePartyId
from party_case_client_details
where CaseStatusName <> 'Cancelled'

######GCOB QUESTION AND ANSWER

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW GCOB_QuestionAnswer AS
Select
  CONCAT('GCOB_', c.UniquePartyId) AS LocalSystemIdentifier,
  c.ClientId,
  c.CaseId,
  CASE
    WHEN c.ClientType like 'Legal%' THEN concat(c.SourceSystem, '_LEC_', c.CaseId)
    WHEN c.ClientType LIKE '%Natural%' THEN concat(c.SourceSystem, '_NP_NPPC_', c.CaseId)
  END as UniqueCaseId,
  rmqa.InstanceId,
  rmqa.QuestionId,
  rmqa.QuestionText,
  rmqa.QuestionCode,
  rmqa.AnswerValue,
  rmqa.AnswerText,
  rmqa.IsMaterial as QuestionIsMaterialFlag,
  'GRAM' as RiskModelSourceSystem
from
  Gcob_CaseClientDetails c
    inner join Party_RiskModelInstanceQuestionAnswers rmqa
      on c.SourceClient = rmqa.SourceClient
where
  rmqa.SourceSystemName = 'GCOB-CaseService'

######LEGACY2 QUESTION AND ANSWER

In [0]:
sdf=spark.table('Legacy2_risk')
columns = [
    "GeographicalRegisteredCountry",
    "GeographicalUboCountries",
    "GeographicalUpCountries",
    "GeographicalHighRiskCountries",
    "EntityTypeCharacterizationReferenceType",
    "EntityTypeCharacterizationTypeText",
    "StructureIssuedBearerShares",
    "StructureMoreThanFourLayers",
    "StructureIsTrust",
    "StructureNomineeShareholders",
    "StructureMultiEntityUltimateBeneficialOwners",
    "StructureNonResidentialMajorityShellShareholder",
    "StructureBearerAlertUsername",
    "StructureBearerAlertDateTime",
    "StructureTrustAndCompanyServiceProvider",
    "StructureTrustAndCompanyServiceProviderExplanation",
    "StructureTrustAndCompanyServiceProviderLogicalLegitimate",
    "StructureTrustAndCompanyServiceProviderLicensedRegulated",
    "StructureUnusualOrComplex",
    "StructureRegistrationIncreasedTaxIntegrityRiskJurisdiction",
    "StructureBroaderOrganizationalChart",
    "SectorBusinessIndustry",
    "SectorBusinessActivitiesDescription",
    "SectorOtherHighRiskActivities",
    "SectorAssociatedRisks",
    "ProductsCommensurateWithClientProfile",
    "ProductsSourceOfRepaymentCommensurate",
    "ProductsAccountPurposeDescription",
    "ProductsExpectedUseDescription",
    "ProductsSourceOfFundsDescription",
    "ProductsSourceOfFundsNADescription",
    "ProductsSourceOfWealthDescription",
    "PoliticallyExposedPersonStatus",
    "TransactionBackToBack",
    "TransactionUnsual",
    "TransactionLoanBack",
    "TransactionFalsifiedRevenues",
    "TransactionNoApparentPurpose",
    "TransactionNonTransparent",
    "TransactionRealEstateRelated",
    "TransactionTaxEvasionFacilitating",
    "TransactionHighRisk",
    "TransactionAnonymousUse",
    "TransactionIsRatioCashElectronicMoney",
    "TransactionRatioCashElectronicMoneyNote",
    "TransactionIsTotalTransactionsInLast12Months",
    "TransactionTotalTransactionsInLast12MonthsNote",
    "TransactionIsTotalTransactionsInLast12MVsLast36Months",
    "TransactionTotalTransactionsInLast12MVsLast36MNote",
    "TransactionIsGeographyOutgoingAndIncomingTransactions",
    "TransactionGeographyOutgoingAndIncomingTransactionsNote",
    "TransactionIsInstrumentsUsedByClientInTransactions",
    "TransactionInstrumentsUsedByClientInTransactionsNote",
    "TransactionIsTotalValueTransactionsInLast12MVsLast36Months",
    "TransactionTotalValueTransactionsInLast12MVsLast36MNote",
    "DistributionFaceToFace",
    "DistributionFaceToFaceStaffName",
    "DistributionFaceToFaceDate",
    "DistributionThirdParty",
    "ThirdPartyActorReferenceId",
    "ThirdPartyCDDbyThirdParty",
    "ThirdPartyWWFT",
    "AdverseInformationTypeReference",
    "OtherRiskNotAnalysed",
    "OtherRiskNotAnalysedComment",
    "OtherLocalLaws",
    "OtherLocalLawsComment",
]

# Create Spark DataFrame by selecting columns from temporary view 'a'
#sdf = spark.sql(f"SELECT {', '.join(columns)} FROM Legacy2_risk")


# Cast all columns to string type
sdf_casted = sdf.select(
    [F.col(c).cast("string").alias(c) for c in sdf.columns]
)

# Convert columns to rows
melted_sdf = sdf_casted.select(
    "ClientId",
    F.explode(
        F.array(
            *[
                F.struct(
                    F.lit(col).alias("QuestionText"), F.col(col).alias("AnswerText")
                )
                for col in columns
            ]
        )
    ).alias("QA"),
).select("ClientId", "QA.QuestionText", "QA.AnswerText")


# Show the result
melted_sdf.createOrReplaceTempView('Legacy2_questions')

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW NonSFDCID_Legacy2_Client AS
select * from Legacy2_case_client_details where isclient = 1 and GcobCaseId is null and Value is null and GcobId like 'RA:%';

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW Legacy2_client AS
select * from Legacy2_case_client_details
where ClientId not in (select ClientId from NonSFDCID_Legacy2_Client)

In [0]:

  %sql
  CREATE OR REPLACE TEMPORARY VIEW Legacy2_QuestionAnswer AS
  Select
 CONCAT('LEGACY2_', CASE
      WHEN IsClient = 'true'
      and ClientTypeId = 1 THEN concat('LE_', c.GcobId)
      WHEN IsClient = 'true'
      and ClientTypeId in (2, 3) THEN concat('NP_NPPC_', c.GcobId)
      WHEN IsClient = 'false'
      and ClientTypeId = 1 THEN concat('RLE_', c.GcobId)
      WHEN IsClient = 'false'
    and ClientTypeId in (2, 3) THEN concat('RNP_', c.GcobId) END) as LocalSystemIdentifier,
    c.ClientId,
    null as CaseId,
    CASE
      WHEN
        c.IsClient = 'true'
        and c.ClientTypeId = 1
      THEN
        concat(c.SourceSystem, '_LEC_', c.ClientId)
      WHEN
        c.IsClient = 'true'
        and c.ClientTypeId in (2, 3)
      THEN
        concat(c.SourceSystem, '_NP_NPPC_', c.ClientId)
      WHEN
        c.IsClient = 'false'
        and c.ClientTypeId = 1
      THEN
        concat(c.SourceSystem, '_RLEP_', c.ClientId)
      WHEN
        c.IsClient = 'false'
        and c.ClientTypeId in (2, 3)
      THEN
        concat(c.SourceSystem, '_RNPP_', c.ClientId)
    END as UniqueCaseId,
    null as InstanceId,
    null as QuestionId,
    rmqa.QuestionText,
    null as QuestionCode,
    null as AnswerValue,
    rmqa.AnswerText,
    null as QuestionIsMaterialFlag
    ,'LEGACY2' as RiskModelSourceSystem
    from
    Legacy2_questions rmqa
    left join Legacy2_client c on c.ClientId = rmqa.ClientId 
  Where 
  c.ClientTypeId in (1,2,3)

In [0]:
 
  # %sql
  # CREATE
  # OR REPLACE TEMPORARY VIEW KN1_QuestionAnswer AS
  # select
  # CONCAT('GIC_',gic.COD_INSTITUCIONAL) as LocalSystemIdentifier,
  #   gic.SEQ_PESSOA as ClientID,
  #   kn_cont.ID_CONTRAPARTE as CaseId,
  #   CASE
  #     WHEN gic.SEQ_TIPO_PESSOA = 2 THEN concat('KN1_NP_NPPC_', CaseId)
  #     ELSE concat('KN1_LE_', CaseId)
  #   END as UniqueCaseId,
  #   gram.InstanceId,
  #   gram.QuestionId,
  #   gram.QuestionText,
  #   gram.QuestionCode,
  #   gram.AnswerValue,
  #   gram.AnswerText
  # from
  #   ADKYC_CONTRAPARTES kn_cont
  #   left join ADRBB_CONTRAPARTES_GRAM stat on kn_cont.ID_CONTRAPARTE = stat.ID_CONTRAPARTE
  #   LEFT join ADKYC_CONTRAPARTES_COMPL kncompl on kn_cont.ID_CONTRAPARTE = kncompl.ID_CONTRAPARTE
  #   INNER JOIN pessoa gic on kncompl.CD_CONTRAPARTE = gic.COD_INSTITUCIONAL
  #   left join Party_RiskModelInstanceQuestionAnswers gram on stat.ID_GRAM_IDENTITY = gram.sourcesystemreferenceid
  #   and SourceSystemName = 'Brazil - KN1'


In [0]:
 
  # %sql
  # CREATE
  # OR REPLACE TEMPORARY VIEW KN1_QuestionAnswer AS
  # select
  # CONCAT('GIC_',gic.COD_INSTITUCIONAL) as LocalSystemIdentifier,
  #   gic.COD_INSTITUCIONAL as ClientID,
  #   kc.CaseId,
  #   CASE
  #     WHEN gic.SEQ_TIPO_PESSOA = 2 THEN concat('KN1_NP_NPPC_', kc.CaseId)
  #     ELSE concat('KN1_LE_', kc.CaseId)
  #   END as UniqueCaseId,
  #   gram.InstanceId,
  #   gram.QuestionId,
  #   gram.QuestionText,
  #   gram.QuestionCode,
  #   gram.AnswerValue,
  #   gram.AnswerText
  # from
  #   ADKYC_CONTRAPARTES kn_cont
  #   Inner join KN1Cases kc on kn_cont.ID_CONTRAPARTE = kc.CaseId
  #   left  join ADRBB_CONTRAPARTES_GRAM stat on kn_cont.ID_CONTRAPARTE = stat.ID_CONTRAPARTE
  #   INNER JOIN pessoa gic on kc.gicid = gic.COD_INSTITUCIONAL
  #   left  join Party_RiskModelInstanceQuestionAnswers gram on stat.ID_GRAM_IDENTITY = gram.InstanceId
  #   and SourceSystemName = 'Brazil - KN1'


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW KN1_QuestionAnswer AS
-- Get all cases with CaseCompletedDate
WITH kn1_cases AS (
  SELECT CaseId AS CaseId, GicID AS ClientId, IsLatestCase, CaseCompletedDate
  FROM KN1Cases
),

-- Get GRAM answers for cases
gram_answers AS (
  SELECT kc.CaseId, kc.ClientId, kc.IsLatestCase, kc.CaseCompletedDate,
         gram.InstanceId, gram.QuestionId, gram.QuestionText,
         gram.QuestionCode, gram.AnswerValue, gram.AnswerText,gram.IsMaterial as QuestionIsMaterialFlag
  FROM kn1_cases kc
  LEFT JOIN ADRBB_CONTRAPARTES_GRAM stat ON kc.CaseId = stat.ID_CONTRAPARTE
  LEFT JOIN Party_RiskModelInstanceQuestionAnswers gram
         ON stat.ID_GRAM_IDENTITY = gram.InstanceId
        AND gram.SourceSystemName = 'Brazil - KN1'
  WHERE gram.InstanceId IS NOT NULL
),

-- Identify cases with GRAM answers
gram_cases AS (
  SELECT DISTINCT CaseId FROM gram_answers
),

-- Get KN1 answers
kn1_answers AS (
  SELECT a.ID_CONTRAPARTE AS CaseId,
         CAST(a.ID_CONTRAPARTE AS STRING) AS InstanceId,
         q.ID_QUESTAO AS QuestionId, q.DE_QUESTAO AS QuestionText,
         q.CD_QUESTAO AS QuestionCode,
         COALESCE(CAST(a.CD_RESPOSTA AS STRING), CAST(a.VL_PONTUACAO_RESP AS STRING)) AS AnswerValue,
         COALESCE(NULLIF(TRIM(a.DE_RESPOSTA_COMPL), ''),
                  NULLIF(TRIM(a.DE_RESPOSTA), ''),
                  CAST(a.CD_RESPOSTA AS STRING)) AS AnswerText
  FROM ADRBB_CONTRAPARTES_QUESTOES_RESPOSTAS a
  LEFT JOIN ADRBB_QUESTOES q ON q.ID_QUESTAO = a.ID_QUESTAO
),

-- KN1 fallback for cases without GRAM
kn1_fallback AS (
  SELECT kc.ClientId, kc.CaseId, kc.IsLatestCase, kc.CaseCompletedDate,
         ka.InstanceId, ka.QuestionId, ka.QuestionText,
         ka.QuestionCode, ka.AnswerValue, ka.AnswerText, null as QuestionIsMaterialFlag
  FROM kn1_cases kc
  LEFT JOIN gram_cases gc ON gc.CaseId = kc.CaseId
  LEFT JOIN kn1_answers ka ON ka.CaseId = kc.CaseId
  WHERE gc.CaseId IS NULL
),

-- UNION all rows, keeping SourceSystem and CaseCompletedDate only within this CTE
finalqueryunion AS (
  SELECT
      CONCAT('GIC_', gic.COD_INSTITUCIONAL) AS LocalSystemIdentifier,
      gic.COD_INSTITUCIONAL AS ClientID,
      g.CaseId, 
      CASE WHEN gic.SEQ_TIPO_PESSOA = 2 THEN CONCAT('KN1_NP_NPPC_', g.CaseId)
           ELSE CONCAT('KN1_LEC_', g.CaseId) END AS UniqueCaseId,
      'GRAM' AS SourceSystem,
      g.InstanceId, g.QuestionId, g.QuestionText,
      g.QuestionCode, g.AnswerValue, g.AnswerText,
      g.IsLatestCase,
      g.CaseCompletedDate,
      g.QuestionIsMaterialFlag
      ,'GRAM' as RiskModelSourceSystem
  FROM gram_answers g
  INNER JOIN pessoa gic ON g.ClientId = gic.COD_INSTITUCIONAL

  UNION ALL
  SELECT
      CONCAT('GIC_', gic.COD_INSTITUCIONAL) AS LocalSystemIdentifier,
      gic.COD_INSTITUCIONAL AS ClientID,
      k.CaseId, 
      CASE WHEN gic.SEQ_TIPO_PESSOA = 2 THEN CONCAT('KN1_NP_NPPC_', k.CaseId)
           ELSE CONCAT('KN1_LEC_', k.CaseId) END AS UniqueCaseId,
      'KN1' AS SourceSystem,
      k.InstanceId, k.QuestionId, k.QuestionText,
      k.QuestionCode, k.AnswerValue, k.AnswerText,
      k.IsLatestCase,
      k.CaseCompletedDate,
      k.QuestionIsMaterialFlag
      ,'KN1' as RiskModelSourceSystem
  FROM kn1_fallback k
  INNER JOIN pessoa gic ON k.ClientId = gic.COD_INSTITUCIONAL
)

-- Final selected columns
SELECT Distinct
    LocalSystemIdentifier,
    ClientID,
    CaseId,
    UniqueCaseId,
    InstanceId,
    QuestionId,
    QuestionText,
    QuestionCode,
    AnswerValue,
    AnswerText,
    QuestionIsMaterialFlag,
    RiskModelSourceSystem
FROM finalqueryunion


######GRAM-RANAZ QUESTION AND ANSWER

In [0]:
# %sql
# CREATE OR REPLACE TEMPORARY VIEW GRAM_RANZ AS
# select distinct 
# CONCAT('RANZ_', split_part(SourceSystemReferenceId, '_', 2)) AS LocalSystemIdentifier
# ,split_part(SourceSystemReferenceId, '_', 2) as ClientId
# ,split_part(SourceSystemReferenceId, '_', 1) as CaseID
# ,CONCAT('RANZ_', split_part(SourceSystemReferenceId, '_', 1)) AS UniqueCaseId
# ,InstanceId
# ,QuestionId
# ,QuestionText
# ,QuestionCode
# ,AnswerValue
# ,AnswerText
# ,IsMaterial as QuestionIsMaterialFlag
# from Party_RiskModelInstanceQuestionAnswers 
# where 
# SourceSystemName = 'RANZ - PegaCdd'
# --and len(SourceSystemReferenceId) >= 15
# and split_part(SourceSystemReferenceId, '_', 2) <> ''

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW GRAM_RANZ AS
select distinct
CONCAT('RANZ_',p.PKEY_SRC_OBJECT) LocalSystemIdentifier,
p.PKEY_SRC_OBJECT as ClientID,
dd.CASE_ID as CaseID,
null as UniqueCaseId, 
dd.GRAM_INSTANCE_ID as InstanceId,
b.QuestionId,
b.QuestionText,
b.QuestionCode,
b.AnswerValue,
b.AnswerText,
b.IsMaterial as QuestionIsMaterialFlag,
'GRAM' as RiskModelSourceSystem
FROM c_b_party as p
JOIN c_b_contr_rol_party as crp
    ON crp.FK_PARTY_ID = p.ROWID_XREF
JOIN c_b_contract as c
    ON c.CONTR_ID = crp.FK_CONTR_ID
LEFT JOIN c_b_due_diligence as dd
    ON dd.FK_CONTR_ID = c.CONTR_ID
left join Party_RiskModelInstanceQuestionAnswers b
on dd.GRAM_INSTANCE_ID = b.InstanceId
 and b.SourceSystemName = 'RANZ - PegaCdd'

%md
##### CREATING A DATAFRAME BY COMBINING ALL THE QUESTION ANSWER INFORMATION FROM DIFFERENT SOURCESYSTEMS AND LOADING THE FINAL DATAOBJECT TO SA RADAR STORAGE ACCOUNT

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW CDD_CaseQuestionAnswers AS
select * FROM GCOB_QuestionAnswer
UNION
select * FROM Legacy2_QuestionAnswer
UNION
select * from KN1_QuestionAnswer
UNION
select * from GRAM_RANZ

In [0]:
df_party_CDDcasequestionanswers=spark.table('CDD_CaseQuestionAnswers')

In [0]:
df_final_party_CDDcasequestionanswers=add_party_identifier(df_party_CDDcasequestionanswers,df_Party_SystemIdentifier)

In [0]:
if RunType == "historical":
    save_to_saradar_storage_account(df_final_party_CDDcasequestionanswers, party_CDDcasequestionanswers_dataobject, radar_datamodel_version_number, environment,Load_Date)
else:
    save_to_saradar_storage_account(df_final_party_CDDcasequestionanswers, party_CDDcasequestionanswers_dataobject, radar_datamodel_version_number, environment)